# Folder 04 / file 02 — scheduled Adult monitor

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n04_ops/n02_monitor.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Profile Adult feature/prediction tables and write monitor_status.gate_ok.


## 1 — Imports


In [ ]:
from datetime import datetime, timezone
from src.n00_shared.dataset import MODEL_FEATURE_COLS
from src.n00_shared.runtime import Settings, configure_mlflow, load_settings, job_run_id


## 2 — `_spark`


In [ ]:
def _spark():
    from pyspark.sql import SparkSession

    return SparkSession.builder.getOrCreate()


## 3 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 4 — `run()` step 1/3


In [ ]:
configure_mlflow(settings)
spark = _spark()
reason = "ok"
gate_ok = True


## 5 — `run()` step 2/3


In [ ]:
try:
    if not spark.catalog.tableExists(settings.feature_fq):
        raise RuntimeError("Adult feature store table missing")
    feats = spark.table(settings.feature_fq)
    missing = [c for c in MODEL_FEATURE_COLS if c not in feats.columns]
    if missing:
        raise RuntimeError(f"Adult feature contract mismatch: {missing}")
    n = feats.count()
    if n < settings.min_table_rows:
        raise RuntimeError(f"feature rows {n} < {settings.min_table_rows}")
    if spark.catalog.tableExists(settings.predictions_fq):
        pred_n = spark.table(settings.predictions_fq).count()
        if pred_n == 0:
            reason = "predictions empty"
            gate_ok = True
    else:
        reason = "predictions table not created yet"
except Exception as exc:
    gate_ok = False
    reason = str(exc)


## 6 — `run()` step 3/3


In [ ]:
status = spark.createDataFrame(
    [
        {
            "as_of": datetime.now(timezone.utc).isoformat(),
            "gate_ok": gate_ok,
            "reason": reason,
            "job_run_id": job_run_id(),
        }
    ]
)
(
    status.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(settings.monitor_fq)
)
print(f"monitor gate_ok={gate_ok} reason={reason}")
if not gate_ok:
    print("enqueue dev train via Jobs API with cooldown (not implemented in-process)")
